<a href="https://colab.research.google.com/github/OdysseusPolymetis/initiation_programmation/blob/main/my_little_robot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mission Python : piloter un robot avec Karel

Ce notebook propose une version modernisée de Karel pour découvrir les bases de Python :

- écrire des instructions ;
- définir une fonction `main()` ;
- répéter des actions ;
- utiliser des conditions ;
- visualiser le robot et le code exécuté ligne par ligne.


## 1. Installation

À lancer une seule fois au début de la session Colab.


In [ ]:
!pip -q install stanfordkarel-notebooks pillow

Vous pouvez trouver les images que j'utilise ici : [images dans le dépôt du cours](https://github.com/OdysseusPolymetis/initiation_programmation/tree/main/images).

Je vous conseille de les télécharger, vous pourrez les réutiliser plus bas. Vous pouvez aussi uploader vos propres images.

## 2. Imports et outils de base


In [ ]:
from stanfordkarel import *

from IPython.display import display, Image as IPythonImage
from PIL import Image, ImageDraw, ImageFont
import io
import math
import sys
import inspect
import html
import traceback
import os

## 3. Rendu personnalisé : robot PNG et objectif PNG

Le principe :

- `utiliser_robot_png("mon_robot.png")` remplace Karel par votre image ;
- `utiliser_goal_png("goal_capsule.png")` remplace les beepers par une image d'objectif de votre choix ;
- `lancer_pas_a_pas(...)` affiche le monde et le code surligné au fur et à mesure.

Conseil : utilise des PNG carrés, avec fond transparent, par exemple `512 × 512`.


La cellule suivante ne sera pas expliquée en détail, mais il faut veiller à l'exécuter.

In [ ]:
_KAREL_CUSTOM = {
    "robot_image": None,
    "robot_scale": 0.82,
    "robot_source_degrees": 0,
    "goal_image": None,
    "goal_scale": 0.78,
}

_ORIGINAL_RENDER_FRAME = None
_ORIGINAL_DRAW_BEEPER = None


def _ouvrir_png(chemin):
    return Image.open(chemin).convert("RGBA")


def _orientation_en_degres(direction):
    orientations = {
        "east": 0,
        "south": 90,
        "west": 180,
        "north": 270,
    }
    if direction not in orientations:
        raise ValueError("Orientation attendue : 'east', 'south', 'west' ou 'north'.")
    return orientations[direction]


def _image_resampling_lanczos():
    return getattr(getattr(Image, "Resampling", Image), "LANCZOS")


def _image_resampling_bicubic():
    return getattr(getattr(Image, "Resampling", Image), "BICUBIC")


def _coller_png_centre(image_de_base, sprite_source, centre_x, centre_y, cell_size, scale, rotation_degres=None):
    sprite = sprite_source.copy()
    taille_max = max(8, int(cell_size * scale))

    sprite.thumbnail((taille_max, taille_max), _image_resampling_lanczos())

    if rotation_degres is not None:
        sprite = sprite.rotate(
            rotation_degres,
            expand=True,
            resample=_image_resampling_bicubic()
        )
        sprite.thumbnail((taille_max, taille_max), _image_resampling_lanczos())

    x = int(centre_x - sprite.width / 2)
    y = int(centre_y - sprite.height / 2)
    image_de_base.paste(sprite, (x, y), sprite)


def _render_frame_personnalise(self):
    from stanfordkarel.karel_constants import DIRECTION_TO_RADIANS

    image = Image.new("RGB", (self.image_width, self.image_height), "white")
    self._frame_image = image
    self.draw = ImageDraw.Draw(image)

    self.draw_world()

    robot = _KAREL_CUSTOM["robot_image"]
    if robot is None:
        self.draw_karel()
    else:
        centre_x = self.calculate_corner_x(self.karel.avenue)
        centre_y = self.calculate_corner_y(self.karel.street)

        angle_cible = math.degrees(DIRECTION_TO_RADIANS[self.karel.direction])
        angle_source = _KAREL_CUSTOM["robot_source_degrees"]

        rotation = -(angle_cible - angle_source)

        _coller_png_centre(
            image,
            robot,
            centre_x,
            centre_y,
            self.cell_size,
            _KAREL_CUSTOM["robot_scale"],
            rotation_degres=rotation
        )

    if hasattr(self, "_frame_image"):
        del self._frame_image

    return image


def _draw_beeper_personnalise(self, location, count):
    goal = _KAREL_CUSTOM["goal_image"]

    if goal is None or not hasattr(self, "_frame_image"):
        return _ORIGINAL_DRAW_BEEPER(self, location, count)

    if count == 0:
        return

    centre_x = self.calculate_corner_x(location[0])
    centre_y = self.calculate_corner_y(location[1])

    _coller_png_centre(
        self._frame_image,
        goal,
        centre_x,
        centre_y,
        self.cell_size,
        _KAREL_CUSTOM["goal_scale"],
        rotation_degres=None
    )

    if count > 1:
        self.draw_text(
            (centre_x, centre_y),
            str(count),
            fill="black",
            font="Arial 12",
            anchor="mm",
            tag="beeper",
        )


def activer_rendu_personnalise():
    global _ORIGINAL_RENDER_FRAME, _ORIGINAL_DRAW_BEEPER

    import stanfordkarel.karel_image_renderer as image_renderer
    import stanfordkarel.karel_renderer as karel_renderer

    if _ORIGINAL_RENDER_FRAME is None:
        _ORIGINAL_RENDER_FRAME = image_renderer.KarelImageRenderer.render_frame

    if _ORIGINAL_DRAW_BEEPER is None:
        _ORIGINAL_DRAW_BEEPER = karel_renderer.KarelBaseRenderer._draw_beeper

    image_renderer.KarelImageRenderer.render_frame = _render_frame_personnalise
    karel_renderer.KarelBaseRenderer._draw_beeper = _draw_beeper_personnalise


def desactiver_rendu_personnalise():
    import stanfordkarel.karel_image_renderer as image_renderer
    import stanfordkarel.karel_renderer as karel_renderer

    if _ORIGINAL_RENDER_FRAME is not None:
        image_renderer.KarelImageRenderer.render_frame = _ORIGINAL_RENDER_FRAME

    if _ORIGINAL_DRAW_BEEPER is not None:
        karel_renderer.KarelBaseRenderer._draw_beeper = _ORIGINAL_DRAW_BEEPER


def utiliser_robot_png(chemin_png, taille=0.82, image_orientee_vers="east"):
    _KAREL_CUSTOM["robot_image"] = _ouvrir_png(chemin_png)
    _KAREL_CUSTOM["robot_scale"] = taille
    _KAREL_CUSTOM["robot_source_degrees"] = _orientation_en_degres(image_orientee_vers)
    activer_rendu_personnalise()


def utiliser_goal_png(chemin_png, taille=0.78):
    _KAREL_CUSTOM["goal_image"] = _ouvrir_png(chemin_png)
    _KAREL_CUSTOM["goal_scale"] = taille
    activer_rendu_personnalise()


def retirer_robot_png():
    _KAREL_CUSTOM["robot_image"] = None


def retirer_goal_png():
    _KAREL_CUSTOM["goal_image"] = None

## 4. Ajouter son propre robot et son goal

Dans Colab, on peut charger des fichiers de `content` avec ces cellules :

```python
from google.colab import files
files.upload()
```

Ensuite, l'image chargée sera renommée en `mon_robot.png` (pareil pour le goal) :

```python
utiliser_robot_png("mon_robot.png", taille=0.85, image_orientee_vers="east")
```


In [ ]:
from google.colab import files
uploaded=files.upload()
ancien_nom = list(uploaded.keys())[0]
nouveau_nom = "mon_robot.png"

os.rename(ancien_nom, nouveau_nom)

print(f"Fichier renommé : {ancien_nom} --> {nouveau_nom}")

In [ ]:
uploaded=files.upload()
ancien_nom = list(uploaded.keys())[0]
nouveau_nom = "goal_capsule.png"

os.rename(ancien_nom, nouveau_nom)

print(f"Fichier renommé : {ancien_nom} --> {nouveau_nom}")

In [ ]:
utiliser_robot_png("mon_robot.png", taille=0.85, image_orientee_vers="east")

In [ ]:
utiliser_goal_png("goal_capsule.png", taille=0.82)

## 6. Lancement classique

Cette fonction simplifie l’appel à Karel.


In [ ]:
def lancer(monde, main, cell_size=58, speed=70):
    run_karel_program(
        world_text=monde,
        main_func=main,
        cell_size=cell_size,
        speed=speed,
    )


## 7. Lancement pas-à-pas avec code surligné

Cette version produit une animation avec :

- le monde Karel à gauche ;
- le code à droite ;
- la ligne active surlignée au moment où le robot exécute une action visible.


In [ ]:
def _police_mono(taille):
    candidats = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf",
        "/usr/share/fonts/truetype/liberation2/LiberationMono-Regular.ttf",
    ]
    for chemin in candidats:
        try:
            return ImageFont.truetype(chemin, taille)
        except OSError:
            pass
    return ImageFont.load_default()


def _police_sans(taille, gras=False):
    candidats = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if gras else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation2/LiberationSans-Bold.ttf" if gras else "/usr/share/fonts/truetype/liberation2/LiberationSans-Regular.ttf",
    ]
    for chemin in candidats:
        try:
            return ImageFont.truetype(chemin, taille)
        except OSError:
            pass
    return ImageFont.load_default()


def _extraire_lignes_de_code(fonctions):
    lignes_affichees = []

    for fonction in fonctions:
        try:
            lignes, debut = inspect.getsourcelines(fonction)
            lignes = [ligne.rstrip("\n") for ligne in lignes]
        except OSError:
            lignes = [
                f"# Source indisponible pour {fonction.__name__}",
                f"{fonction.__name__}()"
            ]
            debut = 1

        if lignes_affichees:
            lignes_affichees.append({
                "texte": "",
                "lineno": None,
                "code_obj": None,
                "fonction": None,
            })

        for i, texte in enumerate(lignes):
            lignes_affichees.append({
                "texte": texte,
                "lineno": debut + i,
                "code_obj": fonction.__code__,
                "fonction": fonction.__name__,
            })

    return lignes_affichees


def _composer_frame(world_image, lignes_code, ligne_active, action="", panneau_code=650):
    marge = 24
    espace = 18
    largeur_monde, hauteur_monde = world_image.size
    largeur = largeur_monde + panneau_code + marge * 2 + espace
    hauteur = max(hauteur_monde + marge * 2, 520)

    img = Image.new("RGB", (largeur, hauteur), "#f7f8fb")
    d = ImageDraw.Draw(img)

    titre = _police_sans(22, gras=True)
    sous_titre = _police_sans(15)
    mono = _police_mono(16)

    x_monde, y_monde = marge, marge
    d.rounded_rectangle(
        (x_monde - 10, y_monde - 10, x_monde + largeur_monde + 10, y_monde + hauteur_monde + 10),
        radius=18,
        fill="white",
        outline="#e5e7eb",
        width=2,
    )
    img.paste(world_image, (x_monde, y_monde))

    # Carte code
    x_code = x_monde + largeur_monde + espace + 10
    y_code = marge - 10
    d.rounded_rectangle(
        (x_code, y_code, x_code + panneau_code, hauteur - marge),
        radius=18,
        fill="#111827",
        outline="#e5e7eb",
        width=2,
    )

    d.text((x_code + 24, y_code + 20), "Code exécuté", font=titre, fill="#f9fafb")
    info = f"Action : {action}" if action else "Départ"
    d.text((x_code + 24, y_code + 52), info, font=sous_titre, fill="#cbd5e1")

    y = y_code + 88
    line_h = 24
    max_lignes = max(1, (hauteur - y - marge) // line_h)

    # Si le code est long, on centre l'affichage autour de la ligne active.
    idx_active = None
    for idx, ligne in enumerate(lignes_code):
        if (
            ligne_active
            and ligne["code_obj"] is ligne_active.get("code_obj")
            and ligne["lineno"] == ligne_active.get("lineno")
        ):
            idx_active = idx
            break

    debut = 0
    if idx_active is not None and len(lignes_code) > max_lignes:
        debut = max(0, idx_active - max_lignes // 2)
        debut = min(debut, max(0, len(lignes_code) - max_lignes))

    lignes_visibles = lignes_code[debut:debut + max_lignes]

    for ligne in lignes_visibles:
        active = (
            ligne_active
            and ligne["code_obj"] is ligne_active.get("code_obj")
            and ligne["lineno"] == ligne_active.get("lineno")
        )

        if active:
            d.rounded_rectangle(
                (x_code + 16, y - 2, x_code + panneau_code - 16, y + line_h - 2),
                radius=8,
                fill="#facc15",
            )
            couleur_ligne = "#111827"
            couleur_num = "#111827"
        else:
            couleur_ligne = "#e5e7eb" if ligne["texte"].strip() else "#6b7280"
            couleur_num = "#93c5fd"

        num = "" if ligne["lineno"] is None else str(ligne["lineno"]).rjust(3)
        d.text((x_code + 24, y), num, font=mono, fill=couleur_num)

        texte = ligne["texte"]
        if len(texte) > 58:
            texte = texte[:55] + "..."

        d.text((x_code + 76, y), texte, font=mono, fill=couleur_ligne)
        y += line_h

    return img


def lancer_pas_a_pas(monde, main, fonctions=None, cell_size=58, speed=70, panneau_code=650):
    from stanfordkarel.karel_program import KarelProgram
    from stanfordkarel.karel_image_renderer import KarelImageRenderer
    from stanfordkarel.student_code import StudentCode
    from stanfordkarel.karel_executor import execute_student_program

    if fonctions is None:
        fonctions = [main]
    else:
        fonctions = list(fonctions)
        if main not in fonctions:
            fonctions.append(main)

    lignes_code = _extraire_lignes_de_code(fonctions)
    objets_code = {fonction.__code__ for fonction in fonctions}
    ligne_active = {"code_obj": None, "lineno": None}

    def trace_lignes(frame, event, arg):
        if event == "line" and frame.f_code in objets_code:
            ligne_active["code_obj"] = frame.f_code
            ligne_active["lineno"] = frame.f_lineno
        return trace_lignes

    karel = KarelProgram(world_text=monde)
    karel.world.init_speed = speed
    renderer = KarelImageRenderer(karel.world, karel, cell_size=cell_size)
    student_code = StudentCode(main_func=main)

    frames = [
        _composer_frame(
            renderer.render_frame(),
            lignes_code,
            ligne_active=None,
            action="départ",
            panneau_code=panneau_code
        )
    ]

    def on_action(action_name):
        frames.append(
            _composer_frame(
                renderer.render_frame(),
                lignes_code,
                ligne_active=ligne_active.copy(),
                action=action_name,
                panneau_code=panneau_code
            )
        )

    ancienne_trace = sys.gettrace()
    success = False
    try:
        sys.settrace(trace_lignes)
        success = execute_student_program(student_code, karel, on_action)
    except Exception:
        traceback.print_exc()
    finally:
        sys.settrace(ancienne_trace)

    if not frames:
        print("Aucune image à afficher.")
        return

    duree = max(80, int(1000 * (1 - speed / 100)))

    with io.BytesIO() as buffer:
        frames[0].save(
            buffer,
            "GIF",
            save_all=True,
            append_images=frames[1:],
            duration=duree,
            loop=0,
        )
        display(IPythonImage(data=buffer.getvalue()))

    if not success:
        print("Attention : le programme s'est arrêté avec une erreur ou n'a pas pu être exécuté complètement.")


# Mission 1 — atteindre l’objectif

Le robot doit avancer jusqu’à l’objectif, puis le ramasser.

Attention à la syntaxe du monde :

```text
Beeper: (3, 1); 1
```

Il faut bien le point-virgule `;` avant le nombre.


In [ ]:
def main():
    move()
    move()
    pick_beeper()


monde = """
Dimension: (5, 5)
Karel: (1, 1); east
Beeper: (3, 1); 1
"""

lancer_pas_a_pas(monde, main, fonctions=[main], speed=35)


# Mission 2 — créer une fonction

On crée une fonction `turn_right()` pour apprendre à réutiliser du code.


In [ ]:
def turn_right():
    turn_left()
    turn_left()
    turn_left()


def main():
    turn_left()
    move()
    move()
    move()
    turn_right()
    move()
    move()
    pick_beeper()


monde = """
Dimension: (5, 5)
Karel: (1, 1); east
Beeper: (3, 4); 1
"""

lancer_pas_a_pas(monde, main, fonctions=[turn_right, main], speed=35)


# Mission 3 — répéter avec une boucle

Le robot avance quatre fois pour atteindre l'objectif. On aura deux types de boucle ici, la boucle `for` et la boucle `while`.


In [ ]:
def main():
    for i in range(4):
        move()
    pick_beeper()


monde = """
Dimension: (6, 4)
Karel: (1, 1); east
Beeper: (5, 1); 1
"""

lancer_pas_a_pas(monde, main, fonctions=[main], speed=35)


In [ ]:
def main():
    while no_beepers_present():
        move()
    pick_beeper()


monde = """
Dimension: (6, 4)
Karel: (1, 1); east
Beeper: (5, 1); 1
"""

lancer_pas_a_pas(monde, main, fonctions=[main], speed=35)

# Mission 4 — condition

Le robot avance tant que la route est libre.  
Puis il ramasse l’objectif s’il est présent.


In [ ]:
def main():
    while front_is_clear():
        move()

    if beepers_present():
        pick_beeper()


monde = """
Dimension: (5, 3)
Karel: (1, 1); east
Beeper: (5, 1); 1
Wall: (5, 1); east
"""

lancer_pas_a_pas(monde, main, fonctions=[main], speed=35)


## Aide-mémoire Karel

Commandes principales :

```python
move()
turn_left()
pick_beeper()
put_beeper()
```

Tests utiles :

```python
front_is_clear()
front_is_blocked()
beepers_present()
no_beepers_present()
facing_east()
facing_north()
```

Structure minimale :

```python
def main():
    move()
```

Lancement :

```python
lancer_pas_a_pas(monde, main)
```
